In [ ]:
#| default_exp machine_learning.ml_eval

> **AI-generated — pending human review.** Created by Claude Code on 2026-10-22. Review checklist: `_nbdev_development/AI_REVIEW_CHECKLIST.md` (workspace root). Remove this cell only after review passes.

Every model decision in the consolidation study (UltraBERT pilot, seq2seq→LoRA, ModernBERT retrains, per-task cutover) is gated on *frozen evals*: a fixed pure dataset per task, a fixed metric, and a recorded baseline number. Without that trio, "the new model is fine" is a vibes swap — and the migration fear ("new module quietly does something different") has no objective answer.

This module is the harness: `freeze_eval_set` writes an immutable, purity-enforced eval set under `_ml_data/eval/`; `save_baseline`/`latest_baseline` keep a per-task history of model-id + metrics; `compare_to_baseline` implements the gate — **each metric must reach baseline − ε, never an average** (the classic multi-task failure is one task's +5 hiding another's −8). Model execution deliberately lives *outside* the harness: `run_eval` takes injected `predict_fn`/`metric_fn`, so importing and testing this module needs no `transformers`, no network, and no pickles — per-task runners (playgrounds now, #19 training modules later) supply them.

In [ ]:
#| export
import json
from datetime import date
from os import PathLike
from pathlib import Path
from typing import Callable

from trouver.machine_learning.ml_data import (AUGMENTATION_COLUMN, ML_DATA_DIR_NAME, PROVENANCE_COLUMN,
                                              PURE_LABEL, DatasetSchema, load_rows, save_rows)

EVAL_DIR_NAME = 'eval'
"""Subdirectory of `<root>/_ml_data` holding frozen eval sets and `baselines.json`,
kept apart from training datasets so an eval row can never be picked up by a
training glob by accident."""

In [ ]:
#| export
def eval_dataset_dir(root: PathLike, schema: DatasetSchema) -> Path:
    """Frozen eval set directory `<root>/_ml_data/eval/<name>_v<N>` for `schema`."""
    return Path(root) / ML_DATA_DIR_NAME / EVAL_DIR_NAME / f'{schema.name}_v{schema.version}'


def baselines_path(root: PathLike) -> Path:
    """JSON file recording every task's baseline history: `<root>/_ml_data/eval/baselines.json`."""
    return Path(root) / ML_DATA_DIR_NAME / EVAL_DIR_NAME / 'baselines.json'


def freeze_eval_set(rows: list[dict], root: PathLike, schema: DatasetSchema, force: bool = False) -> Path:
    """
    Validate `rows` against `schema` and write the frozen eval set; return its path.

    Refuses any row labeled augmented (frozen eval sets are pure by rule R5) and
    refuses to overwrite an existing set — *frozen* means the number a baseline
    was measured on still exists byte-identically. To change an eval set, bump
    the schema version; `force=True` is only for correcting a set caught broken
    before any baseline referenced it.

    **Parameters**
    - `rows` (*list[dict]*) — Pure eval rows, already split out from training data by note name.
    - `root` (*PathLike*) — Vault/dataset root holding `_ml_data/`.
    - `schema` (*DatasetSchema*) — The task's dataset schema.
    - `force` (*bool*) — Replace an existing frozen set (discouraged; see summary).
    """
    for row in rows:
        label = row.get(AUGMENTATION_COLUMN, PURE_LABEL)
        if label != PURE_LABEL:
            raise ValueError(f'Eval set must be pure; found {AUGMENTATION_COLUMN}={label!r}.')
    schema.validate_rows(rows)
    path = eval_dataset_dir(root, schema) / f'{schema.name}.jsonl'
    if path.exists() and not force:
        raise ValueError(f'{path} is frozen; bump the schema version instead of overwriting.')
    path.parent.mkdir(parents=True, exist_ok=True)
    save_rows(rows, path)
    return path


def load_eval_rows(root: PathLike, schema: DatasetSchema) -> list[dict]:
    """Load the frozen eval set for `schema`, re-enforcing purity on every read."""
    return load_rows(eval_dataset_dir(root, schema) / f'{schema.name}.jsonl',
                     schema=schema, require_pure=True)

In [ ]:
#| export
def check_split_leakage(eval_rows: list[dict], *row_lists: list[dict]) -> None:
    """
    Raise `ValueError` listing notes shared between the eval rows and any training list.

    The note-name-keyed split (`machine_learning.train_split`) keeps these
    disjoint by construction; this is the loud tripwire for a producer or an
    augmentation run that broke the rule — leakage found here costs a rerun,
    found in a published metric it costs trust.

    **Parameters**
    - `eval_rows` (*list[dict]*) — The frozen eval set.
    - `row_lists` (*list[dict]*) — Any number of training row lists to check against.
    """
    eval_notes = {r[PROVENANCE_COLUMN] for r in eval_rows}
    for i, rows in enumerate(row_lists):
        overlap = sorted(eval_notes & {r[PROVENANCE_COLUMN] for r in rows})
        if overlap:
            raise ValueError(f'Row list {i} shares notes with the eval set: {overlap}.')

In [ ]:
#| export
def load_baselines(root: PathLike) -> dict:
    """Load `{task: [record, ...]}` (oldest first) from `baselines_path(root)`; `{}` before the first baseline."""
    path = baselines_path(root)
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}


def save_baseline(root: PathLike, task: str, model_id: str, metrics: dict, dataset: str = None) -> dict:
    """
    Append one baseline record for `task` and return it.

    History is append-only: superseded baselines stay, so an old gate verdict
    can always be replayed against the numbers it was actually made with.

    **Parameters**
    - `root` (*PathLike*) — Vault/dataset root holding `_ml_data/`.
    - `task` (*str*) — Task key, e.g. `'def_notat_ner'` (registry naming).
    - `model_id` (*str*) — HF repo id or artifact path the metrics were measured with.
    - `metrics` (*dict*) — `{metric_name: float}` exactly as the gate should compare it.
    - `dataset` (*str*) — Frozen eval set id (e.g. `'def_notat_eval_v1'`) the run used.
    """
    record = {'date': date.today().isoformat(), 'task': task, 'model_id': model_id,
              'dataset': dataset, 'metrics': dict(metrics)}
    data = load_baselines(root)
    data.setdefault(task, []).append(record)
    path = baselines_path(root)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=1, ensure_ascii=False), encoding='utf-8')
    return record


def latest_baseline(root: PathLike, task: str) -> dict:
    """Return the newest baseline record for `task`; `KeyError` listing known tasks when none exists."""
    data = load_baselines(root)
    if not data.get(task):
        raise KeyError(f'No baseline recorded for task {task!r}. Known tasks: {sorted(data)}.')
    return data[task][-1]

In [ ]:
#| export
def compare_to_baseline(metrics: dict, baseline_metrics: dict, epsilon: float = 0.0) -> dict:
    """
    Gate verdict `{'passed': bool, 'details': {name: {...}}}`: each metric must be >= baseline - `epsilon`.

    Metrics are judged one at a time and **never averaged** — a multi-task
    model may not trade one task's regression for another's gain. Metric-name
    mismatches in either direction raise `ValueError`: comparing different
    measuring sticks is not a gate, it is noise.

    **Parameters**
    - `metrics` (*dict*) — `{name: float}` from the candidate run.
    - `baseline_metrics` (*dict*) — `{name: float}` from the baseline record.
    - `epsilon` (*float*) — Tolerance subtracted from each baseline value.
    """
    missing = sorted(set(baseline_metrics) - set(metrics))
    if missing:
        raise ValueError(f'Candidate run is missing baseline metrics: {missing}.')
    extra = sorted(set(metrics) - set(baseline_metrics))
    if extra:
        raise ValueError(f'Candidate reports metrics the baseline never measured: {extra}.')
    details = {name: {'value': metrics[name], 'baseline': base,
                      'passed': metrics[name] >= base - epsilon}
               for name, base in baseline_metrics.items()}
    return {'passed': all(d['passed'] for d in details.values()), 'details': details}


def run_eval(eval_rows: list[dict], predict_fn: Callable, metric_fn: Callable) -> dict:
    """
    Score `predict_fn` on frozen `eval_rows`; returns `metric_fn(eval_rows, predictions)`.

    Model loading (HF `from_pretrained`, `fastai_bridge.load_fastai_learner`)
    lives inside the caller's `predict_fn` — the harness itself stays free of
    `transformers`/`fastai`/network so it imports and tests anywhere.

    **Parameters**
    - `eval_rows` (*list[dict]*) — Frozen pure rows (from `load_eval_rows`).
    - `predict_fn` (*Callable*) — `predict_fn(rows) -> list`, one prediction per row, same order.
    - `metric_fn` (*Callable*) — `metric_fn(rows, predictions) -> dict[str, float]`; wrap `training_helpers` metrics or an exact-match score here.
    """
    predictions = predict_fn(eval_rows)
    if len(predictions) != len(eval_rows):
        raise ValueError(f'predict_fn returned {len(predictions)} predictions for {len(eval_rows)} rows.')
    return metric_fn(eval_rows, predictions)


def evaluate_and_gate(root: PathLike, task: str, eval_rows: list[dict], predict_fn: Callable,
                      metric_fn: Callable, epsilon: float = 0.0) -> dict:
    """
    `run_eval` + gate against `latest_baseline(root, task)` in one call.

    **Returns**
    `{'metrics', 'passed', 'details', 'baseline'}` — the full record of one
    gate decision; save it next to the run that produced it.
    """
    metrics = run_eval(eval_rows, predict_fn, metric_fn)
    baseline = latest_baseline(root, task)
    verdict = compare_to_baseline(metrics, baseline['metrics'], epsilon=epsilon)
    return {'metrics': metrics, 'baseline': baseline, **verdict}

In [ ]:
import tempfile
from pathlib import Path

from fastcore.test import *

from trouver.machine_learning import ml_data as mld

In [ ]:
# Freeze: pure rows accepted, augmented refused, overwrite refused, load re-checks purity.
schema = mld.DatasetSchema('demo_eval', 1, ('Note name', 'text'))
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    rows = [{'Note name': 'a', 'text': 'x'}, {'Note name': 'b', 'text': 'y'}]
    path = freeze_eval_set(rows, root, schema)
    assert path.exists() and path.parent == eval_dataset_dir(root, schema)
    test_eq(load_eval_rows(root, schema), rows)

    with ExceptionExpected(regex='must be pure'):
        freeze_eval_set([{'Note name': 'a', 'text': 'x', mld.AUGMENTATION_COLUMN: 'latex_robust_v1'}], root, schema)
    with ExceptionExpected(regex='frozen'):
        freeze_eval_set(rows, root, schema)
    force_path = freeze_eval_set(rows, root, schema, force=True)  # explicit escape hatch works
    assert force_path.exists()

    # A tampered on-disk set (augmented row written behind the harness's back) is caught at load.
    bad_path = eval_dataset_dir(root, schema) / 'demo_eval.jsonl'
    mld.save_rows([{'Note name': 'a', 'text': 'x', mld.AUGMENTATION_COLUMN: 'latex_noise_v1'}], bad_path)
    with ExceptionExpected(regex='require_pure'):
        load_eval_rows(root, schema)

In [ ]:
# Leakage tripwire: shared note names are named in the error; disjoint sets pass silently.
evalr = [{'Note name': 'a'}, {'Note name': 'b'}]
check_split_leakage(evalr, [{'Note name': 'c'}], [{'Note name': 'd'}])
with ExceptionExpected(regex=r"shares notes"):
    check_split_leakage(evalr, [{'Note name': 'b'}, {'Note name': 'z'}])

In [ ]:
# Baselines: append-only history per task, latest wins, unknown task names known tasks.
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    test_eq(load_baselines(root), {})
    save_baseline(root, 'ner', 'old_model', {'f1': 0.80}, dataset='demo_eval_v1')
    save_baseline(root, 'ner', 'new_model', {'f1': 0.83}, dataset='demo_eval_v1')
    data = load_baselines(root)
    test_eq(len(data['ner']), 2)
    test_eq(latest_baseline(root, 'ner')['model_id'], 'new_model')
    test_eq(latest_baseline(root, 'ner')['dataset'], 'demo_eval_v1')
    try:
        latest_baseline(root, 'linking')
        assert False
    except KeyError as e:
        assert 'ner' in str(e)  # error lists what IS known

In [ ]:
# Gate: per-metric verdicts, epsilon, never-averaged, loud on measuring-stick mismatch.
v = compare_to_baseline({'f1': 0.82, 'acc': 0.90}, {'f1': 0.80, 'acc': 0.95})
assert not v['passed']  # acc regressed even though f1 improved — averaging would have hidden it
test_eq(v['details']['acc'], {'value': 0.90, 'baseline': 0.95, 'passed': False})
assert compare_to_baseline({'f1': 0.82}, {'f1': 0.80}, epsilon=0.03)['passed']
with ExceptionExpected(regex='missing baseline metrics'):
    compare_to_baseline({'f1': 0.9}, {'f1': 0.9, 'acc': 0.9})
with ExceptionExpected(regex='never measured'):
    compare_to_baseline({'f1': 0.9, 'bleu': 0.5}, {'f1': 0.9})

In [ ]:
# run_eval + evaluate_and_gate end to end with injected fakes (no model, no network).
rows = [{'Note name': 'a', 'label': 1}, {'Note name': 'b', 'label': 0}]
acc = lambda rws, preds: {'acc': sum(p == r['label'] for r, p in zip(rws, preds)) / len(rws)}
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    test_eq(run_eval(rows, lambda rws: [1, 0], acc), {'acc': 1.0})
    with ExceptionExpected(regex='predictions for'):
        run_eval(rows, lambda rws: [1], acc)

    save_baseline(root, 'demo_task', 'm0', {'acc': 0.5})
    out = evaluate_and_gate(root, 'demo_task', rows, lambda rws: [1, 0], acc)
    assert out['passed'] and out['metrics'] == {'acc': 1.0}
    test_eq(out['baseline']['model_id'], 'm0')

## Examples

In [ ]:
# A gate decision for a hypothetical NER task, end to end (fake predict/metric stand in for the model).
schema = mld.DatasetSchema('def_notat_eval', 1, ('Note name', 'tokens', 'labels'))
eval_rows = [{'Note name': 'My Theorem', 'tokens': ['Let', '$x$'], 'labels': ['O', 'B-def']}]
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    freeze_eval_set(eval_rows, root, schema)
    save_baseline(root, 'def_notat_ner', 'hyunjongkimmath/def_and_notat_token_classification_model_modernbert_base',
                  {'overall_f1': 0.91}, dataset='def_notat_eval_v1')

    def predict(rows):  # stands in for tokenizer + model + postprocessing
        return [['O', 'B-def'] for _ in rows]
    def metric(rows, preds):
        agree = sum(p == l for rws, prd in zip(rows, preds) for p, l in zip(rws['labels'], prd))
        return {'overall_f1': agree / sum(len(r['labels']) for r in rows)}

    verdict = evaluate_and_gate(root, 'def_notat_ner', load_eval_rows(root, schema), predict, metric, epsilon=0.01)
    assert verdict['passed'] and verdict['details']['overall_f1']['baseline'] == 0.91